# 2 µm ink inference benchmark (Colab)

A thin wrapper around `python -m bench`. It installs the pinned environment,
runs W0 and W1 with random weights (no downloads), and shows the results.
Nothing is uploaded anywhere: the harness guard blocks AWS clients and
unexpected network hosts. Download the JSON files at the end by hand.

Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
REPO = "https://github.com/d-wasserman/villa"
BRANCH = "claude/gracious-noether-6l33ee"
TARGET_TAG = "colab-t4"
BATCH_SIZES = [1, 2, 4, 8, 16]   # W0 sweep; stops at the first out-of-memory
W1_BATCH = 4
W1_WORKERS = 2                   # Colab has 2 vCPUs
REPEATS = 5

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!rm -rf villa && git clone --depth 1 --branch {BRANCH} --filter=blob:none --sparse {REPO} villa
!cd villa && git sparse-checkout set ink-detection/optimized_inference
%cd villa/ink-detection/optimized_inference
!git log -1 --oneline

Pin torch 2.10 (CUDA 12.8) to match the production Dockerfile; Colab ships
its own version. `nvidia-ml-py` gives GPU temperature, clocks and utilization.
If Colab asks to restart the runtime after this cell, do so, then re-run the
first two cells (settings and `%cd`) and continue from the next one.

In [ ]:
!pip install -q "torch==2.10.*" --index-url https://download.pytorch.org/whl/cu128
!pip install -q -r requirements-cpu-only.txt setuptools nvidia-ml-py
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
!python -m unittest discover -s tests -t . 2>&1 | tail -3
!python -m bench flops

## W0: forward pass only, batch-size sweep (production numerics: fp16 autocast)

In [ ]:
import subprocess
for bs in BATCH_SIZES:
    r = subprocess.run(["python", "-m", "bench", "run", "--workload", "W0", "--target-tag", TARGET_TAG,
                        "--batch-size", str(bs), "--w0-iters", "5", "--repeats", str(REPEATS),
                        "--tag", f"bs{bs}"], capture_output=True, text=True)
    print(f"batch {bs}:", r.stdout[-600:] if r.returncode == 0 else "FAILED\n" + r.stderr[-800:])
    if r.returncode != 0 and "out of memory" in r.stderr.lower():
        break

## W1: full inference loop, throughput then attribution, compile off / reduce-overhead

In [ ]:
for compile_mode in ["off", "reduce-overhead"]:
    for mode in ["throughput", "attribution"]:
        !python -m bench run --workload W1 --target-tag {TARGET_TAG} --mode {mode} \
            --batch-size {W1_BATCH} --workers {W1_WORKERS} --compile {compile_mode} \
            --inductor-cache cold --repeats {REPEATS} --tag compile-{compile_mode} 2>&1 | tail -12

## Results

In [ ]:
import glob, json
import pandas as pd
rows = []
for f in sorted(glob.glob(f"bench/results/{TARGET_TAG}/*.json")):
    d = json.load(open(f)); c, t, g = d["config"], d["timings"], d["telemetry"]
    rows.append({
        "file": f.split("/")[-1], "workload": c["workload"], "mode": c["mode"],
        "batch": c["batch_size"], "compile": c["compile_mode"],
        "s/tile": t["seconds_per_tile"]["mean"], "cv": t["seconds_per_tile"]["cv"],
        "s/cm2 gross": t["device_seconds_per_cm2_gross"]["mean"],
        "first batch s": t["first_batch_seconds"], "compile s": t["compile_seconds"],
        "stage coverage": t["stage_coverage"], "gpu util %": g["gpu_utilization_percent_avg"],
        "max temp C": g["gpu_temperature_c_max"], "min SM MHz": g["gpu_sm_clock_mhz_min"],
        "peak VRAM GB": (g["peak_vram_bytes"] or 0) / 1e9,
    })
pd.DataFrame(rows)

In [ ]:
# Stage breakdown for attribution runs
for f in sorted(glob.glob(f"bench/results/{TARGET_TAG}/*attribution*.json")):
    d = json.load(open(f))
    print(f.split("/")[-1], "accepted:", d["timings"]["stage_breakdown_accepted"])
    print(pd.Series(d["timings"]["stage_seconds"]).round(3).to_string(), "\n")

In [ ]:
# Download the result files (small JSON summaries) to commit to the fork
!cd bench/results && zip -qr /content/results-{TARGET_TAG}.zip {TARGET_TAG}
from google.colab import files
files.download(f"/content/results-{TARGET_TAG}.zip")